## 7. Summary: Hybrid Recommendation System\n\nThis notebook demonstrates the four-layer hybrid approach outlined in the proposal:\n\n### ✅ **Layer 1: Collaborative Filtering**\n- Uses SVD matrix factorization on student-book interaction data\n- Finds books borrowed by students with similar reading patterns\n- Primary behavioral signal with proven scalability\n\n### ✅ **Layer 2: Content-Based Similarity**  \n- Uses TF-IDF vectors from book descriptions, genres, and reading levels\n- Handles cold start cases for new students or new books\n- Provides discovery beyond just popular titles\n\n### ✅ **Layer 3: Policy Guardrails**\n- Enforces grade-band appropriateness \n- Respects district suitability constraints\n- Allows flexibility for advanced readers\n\n### ✅ **Layer 4: LLM Explanations**\n- Generates understandable \"because you liked X, try Y\" explanations\n- Makes recommendations transparent to students and librarians\n- Builds trust without letting LLM control ranking\n\n### **Next Steps for Full Implementation:**\n- Integrate with real Destiny Library Manager exports\n- Add librarian review workspace for approval/override\n- Build student-facing UI with ClassLink SSO\n- Add district dashboard for pilot metrics\n- Deploy as FastAPI service with proper authentication\n\n**This hybrid approach balances recommendation quality, policy compliance, and user trust - exactly what Fulton County needs for their Reading Lift pilot.**"

In [ ]:
def generate_explanation(student_id, book_id, recommendation_methods):\n    \"\"\"Simulate LLM-generated explanations for recommendations\"\"\"\n    \n    # Get student reading history\n    student_history = checkouts_df[checkouts_df['student_id'] == student_id]['book_id'].tolist()\n    recommended_book = catalog_df[catalog_df['book_id'] == book_id].iloc[0]\n    \n    if len(student_history) == 0:\n        return f\"Since you're new to our library, we think you'll enjoy {recommended_book['title']}, a popular {recommended_book['genre'].lower()} book.\"\n    \n    # Get a representative book from their history\n    recent_book_id = student_history[-1]  # Most recent\n    recent_book = catalog_df[catalog_df['book_id'] == recent_book_id].iloc[0]\n    \n    # Generate explanation based on recommendation method\n    if 'collaborative_filtering' in recommendation_methods:\n        return f\"Students who enjoyed {recent_book['title']} also loved {recommended_book['title']}. It's a {recommended_book['genre'].lower()} story that readers like you found engaging.\"\n    \n    elif 'content_based' in recommendation_methods:\n        if recent_book['genre'] == recommended_book['genre']:\n            return f\"Since you enjoyed the {recent_book['genre'].lower()} elements in {recent_book['title']}, you might like {recommended_book['title']}, another {recommended_book['genre'].lower()} story.\"\n        else:\n            return f\"Based on your interest in {recent_book['title']}, we think you'll enjoy {recommended_book['title']} for its similar themes and reading level.\"\n    \n    else:\n        return f\"We recommend {recommended_book['title']} as a great {recommended_book['genre'].lower()} book for your grade level.\"\n\n# Demonstrate explanations\nprint(\"Sample LLM-Generated Explanations\")\nprint(\"==================================\")\n\n# Get recommendations for our heavy reader\nmost_active = checkouts_df['student_id'].value_counts().index[0]\nrecs = hybrid_system.get_hybrid_recommendations(most_active, students_df, n_recommendations=3)\n\nfor i, rec in enumerate(recs, 1):\n    book_info = catalog_df[catalog_df['book_id'] == rec['book_id']].iloc[0]\n    explanation = generate_explanation(most_active, rec['book_id'], rec.get('methods', []))\n    \n    print(f\"\\nRecommendation {i}: {book_info['title']}\")\n    print(f\"Explanation: {explanation}\")\n    print(f\"Confidence: {rec['score']:.3f}\")"

## 6. LLM Explanation Layer (Simulated)\n\nIn the full system, LLMs would generate explanations like \"Because you enjoyed thoughtful dystopian stories like The Giver, you might like Scythe.\" Here's how that would work:"

In [ ]:
def demonstrate_recommendations(student_id, title="Student Recommendations"):
    print(f"\\n=== {title} ===")
    
    # Get student info
    student_info = students_df[students_df['student_id'] == student_id].iloc[0]
    print(f"Student: {student_id}")
    print(f"Grade: {student_info['grade']}, School: {student_info['school']}")
    print(f"Reading Preference: {student_info['reading_preference']}")
    
    # Show reading history
    history = checkouts_df[checkouts_df['student_id'] == student_id]['book_id'].tolist()
    print(f"\\nReading History ({len(history)} books):")
    for book_id in history[:5]:  # Show first 5
        book_title = catalog_df[catalog_df['book_id'] == book_id]['title'].iloc[0]
        book_genre = catalog_df[catalog_df['book_id'] == book_id]['genre'].iloc[0]
        print(f"  - {book_title} ({book_genre})")
    if len(history) > 5:
        print(f"  ... and {len(history) - 5} more")
    
    if len(history) == 0:
        print("  No reading history (cold start case)")
        return
    
    # Get hybrid recommendations
    try:
        recommendations = hybrid_system.get_hybrid_recommendations(student_id, students_df, n_recommendations=5)
        
        print(f"\\nHybrid Recommendations:")
        if len(recommendations) == 0:
            print("  No recommendations available")
            return
            
        for i, rec in enumerate(recommendations, 1):
            book_info = catalog_df[catalog_df['book_id'] == rec['book_id']].iloc[0]
            methods = ', '.join(rec.get('methods', ['unknown']))
            print(f\"  {i}. {book_info['title']} by {book_info['author']}\")
            print(f\"     Genre: {book_info['genre']}, Grade: {book_info['grade_band']}")
            print(f\"     Score: {rec['score']:.3f}, Methods: {methods}")
            
    except Exception as e:
        print(f\"Error generating recommendations: {e}\")

# Test with students who have different amounts of reading history
active_students = checkouts_df['student_id'].value_counts()

print(\"Testing Hybrid Recommendation System\")
print(\"====================================\")

# Active reader
most_active = active_students.index[0]
demonstrate_recommendations(most_active, \"Heavy Reader\")

# Moderate reader  
moderate_active = active_students.index[len(active_students)//2]
demonstrate_recommendations(moderate_active, \"Moderate Reader\")

# Light reader
light_active = active_students.index[-5]  # One of the less active readers
demonstrate_recommendations(light_active, \"Light Reader\")"

## 5. Demo: Hybrid Recommendations in Action

Let's test the system with different student profiles to show how it works.

In [ ]:
class HybridRecommendationSystem:
    def __init__(self, cf_model, content_model, catalog_df):
        self.cf_model = cf_model
        self.content_model = content_model
        self.catalog_df = catalog_df
        
    def apply_policy_guardrails(self, student_id, recommendations, students_df):
        """Apply grade-band and district policy constraints"""
        student_info = students_df[students_df['student_id'] == student_id].iloc[0]
        student_grade = student_info['grade']
        
        filtered_recommendations = []
        
        for rec in recommendations:
            book_info = self.catalog_df[self.catalog_df['book_id'] == rec['book_id']].iloc[0]
            
            # Grade band check
            grade_band = book_info['grade_band']
            grade_appropriate = False
            
            # Parse grade band (e.g., '6-7', '7-8', '6-8')
            if '-' in grade_band:
                min_grade, max_grade = map(int, grade_band.split('-'))
                if min_grade <= student_grade <= max_grade:
                    grade_appropriate = True
            else:
                if int(grade_band) == student_grade:
                    grade_appropriate = True
            
            # Allow some flexibility for advanced readers
            if not grade_appropriate and book_info['reading_level'] == 'Below Grade':
                grade_appropriate = True
            
            if grade_appropriate:
                rec['grade_appropriate'] = True
                filtered_recommendations.append(rec)
            else:
                rec['grade_appropriate'] = False
                rec['filtered_reason'] = f\"Grade band {grade_band} not suitable for grade {student_grade}\"\n        
        return filtered_recommendations
    
    def get_hybrid_recommendations(self, student_id, students_df, n_recommendations=5):
        \"\"\"Generate hybrid recommendations combining CF and content-based approaches\"\"\"\n        \n        # Get recommendations from both models\n        cf_recs = self.cf_model.get_recommendations(student_id, n_recommendations * 2)\n        content_recs = self.content_model.get_recommendations(\n            student_id, checkouts_df, self.catalog_df, n_recommendations * 2\n        )\n        \n        # Combine and weight recommendations\n        combined_recs = {}\n        \n        # Weight collaborative filtering higher (0.6)\n        for rec in cf_recs:\n            book_id = rec['book_id']\n            if book_id not in combined_recs:\n                combined_recs[book_id] = {'book_id': book_id, 'score': 0, 'methods': []}\n            combined_recs[book_id]['score'] += rec['score'] * 0.6\n            combined_recs[book_id]['methods'].append('collaborative_filtering')\n        \n        # Weight content-based lower (0.4)\n        for rec in content_recs:\n            book_id = rec['book_id']\n            if book_id not in combined_recs:\n                combined_recs[book_id] = {'book_id': book_id, 'score': 0, 'methods': []}\n            combined_recs[book_id]['score'] += rec['score'] * 0.4\n            combined_recs[book_id]['methods'].append('content_based')\n        \n        # Sort by combined score\n        final_recs = list(combined_recs.values())\n        final_recs.sort(key=lambda x: x['score'], reverse=True)\n        \n        # Apply policy guardrails\n        filtered_recs = self.apply_policy_guardrails(student_id, final_recs, students_df)\n        \n        return filtered_recs[:n_recommendations]\n\n# Create hybrid system\nhybrid_system = HybridRecommendationSystem(cf_model, content_model, catalog_df)\nprint(\"Hybrid recommendation system initialized\")"

## 4. Policy Guardrails & Hybrid System

Combine collaborative filtering and content-based recommendations with district policy constraints.

In [ ]:
class ContentBasedRecommender:
    def __init__(self):
        self.tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
        self.content_matrix = None
        self.book_to_idx = {}
        self.idx_to_book = {}
    
    def fit(self, catalog_df):
        # Create content features from description + genre + reading level
        content_features = []
        for _, book in catalog_df.iterrows():
            features = f"{book['description']} {book['genre']} {book['reading_level']}"
            content_features.append(features)
        
        # Build TF-IDF matrix
        self.content_matrix = self.tfidf.fit_transform(content_features)
        
        # Create mappings
        self.book_to_idx = {book_id: idx for idx, book_id in enumerate(catalog_df['book_id'])}
        self.idx_to_book = {idx: book_id for book_id, idx in self.book_to_idx.items()}
        
        return self
    
    def get_recommendations(self, student_id, checkouts_df, catalog_df, n_recommendations=10):
        # Get student's reading history
        student_books = checkouts_df[checkouts_df['student_id'] == student_id]['book_id'].unique()
        
        if len(student_books) == 0:
            return []  # Cold start - no history
        
        # Calculate average content profile from student's history
        student_profile = np.zeros(self.content_matrix.shape[1])
        valid_books = 0
        
        for book_id in student_books:
            if book_id in self.book_to_idx:
                book_idx = self.book_to_idx[book_id]
                student_profile += self.content_matrix[book_idx].toarray().flatten()
                valid_books += 1
        
        if valid_books == 0:
            return []\n        
        student_profile = student_profile / valid_books
        
        # Calculate similarity with all books
        similarities = cosine_similarity([student_profile], self.content_matrix).flatten()
        
        # Remove already borrowed books
        for book_id in student_books:
            if book_id in self.book_to_idx:
                book_idx = self.book_to_idx[book_id]
                similarities[book_idx] = -1
        
        # Get top recommendations
        top_indices = np.argsort(similarities)[::-1][:n_recommendations]
        
        recommendations = []
        for idx in top_indices:
            book_id = self.idx_to_book[idx]
            score = float(similarities[idx])
            if score > 0:  # Valid recommendation
                recommendations.append({
                    'book_id': book_id,
                    'score': score,
                    'method': 'content_based'
                })
        
        return recommendations

# Train content-based model
content_model = ContentBasedRecommender()
content_model.fit(catalog_df)
print(f"Content-based model trained on {len(content_model.book_to_idx)} books")

## 3. Content-Based Similarity Engine

Recommend books with related themes, genres, and characteristics.

In [ ]:
class CollaborativeFilteringRecommender:
    def __init__(self, n_components=20):
        self.n_components = n_components
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)
        self.user_item_matrix = None
        self.user_factors = None
        self.item_factors = None
        self.user_to_idx = {}
        self.item_to_idx = {}
        self.idx_to_user = {}
        self.idx_to_item = {}
    
    def fit(self, checkouts_df):
        # Create user-item interaction matrix
        users = checkouts_df['student_id'].unique()
        items = checkouts_df['book_id'].unique()
        
        self.user_to_idx = {user: idx for idx, user in enumerate(users)}
        self.item_to_idx = {item: idx for idx, item in enumerate(items)}
        self.idx_to_user = {idx: user for user, idx in self.user_to_idx.items()}
        self.idx_to_item = {idx: item for item, idx in self.item_to_idx.items()}
        
        # Build matrix
        n_users, n_items = len(users), len(items)
        matrix = np.zeros((n_users, n_items))
        
        for _, row in checkouts_df.iterrows():
            user_idx = self.user_to_idx[row['student_id']]
            item_idx = self.item_to_idx[row['book_id']]
            matrix[user_idx, item_idx] = 1  # Binary interaction
        
        self.user_item_matrix = csr_matrix(matrix)
        
        # Apply SVD for dimensionality reduction
        self.user_factors = self.svd.fit_transform(matrix)
        self.item_factors = self.svd.components_.T
        
        return self
    
    def get_recommendations(self, student_id, n_recommendations=10):
        if student_id not in self.user_to_idx:
            return []  # Cold start - no history
        
        user_idx = self.user_to_idx[student_id]
        user_vector = self.user_factors[user_idx]
        
        # Compute scores for all items
        scores = np.dot(user_vector, self.item_factors.T)
        
        # Remove already borrowed books
        borrowed_items = self.user_item_matrix[user_idx].nonzero()[1]
        scores[borrowed_items] = -np.inf
        
        # Get top recommendations
        top_items = np.argsort(scores)[::-1][:n_recommendations]
        
        recommendations = []
        for item_idx in top_items:
            book_id = self.idx_to_item[item_idx]
            score = scores[item_idx]
            if score > -np.inf:  # Valid recommendation
                recommendations.append({
                    'book_id': book_id,
                    'score': float(score),
                    'method': 'collaborative_filtering'
                })
        
        return recommendations

# Train collaborative filtering model
cf_model = CollaborativeFilteringRecommender()
cf_model.fit(checkouts_df)
print(f"Collaborative filtering model trained on {len(cf_model.user_to_idx)} students and {len(cf_model.item_to_idx)} books")

## 2. Collaborative Filtering Engine

Find books borrowed by students with similar reading patterns.

In [ ]:
# Generate synthetic students and checkout history
n_students = 150  # Across 3 schools
n_checkouts = 800  # Total checkout events

schools = ['Mill Creek MS', 'Woodland MS', 'River Trail MS']
grades = [6, 7, 8]

# Create students
students_data = []
for i in range(n_students):
    student = {
        'student_id': f'student_{i+1:03d}',
        'grade': np.random.choice(grades),
        'school': np.random.choice(schools),
        'reading_preference': np.random.choice(['Fantasy', 'Adventure', 'Realistic Fiction', 'Mystery'])
    }
    students_data.append(student)

students_df = pd.DataFrame(students_data)

# Generate realistic checkout patterns
checkouts_data = []
for _ in range(n_checkouts):
    student = students_df.sample(1).iloc[0]
    
    # Bias toward student's reading preference and grade-appropriate books
    suitable_books = catalog_df[
        (catalog_df['grade_band'].str.contains(str(student['grade']))) |
        (catalog_df['genre'] == student['reading_preference'])
    ]
    
    if len(suitable_books) == 0:
        suitable_books = catalog_df
    
    book = suitable_books.sample(1).iloc[0]
    
    checkout = {
        'student_id': student['student_id'],
        'book_id': book['book_id'],
        'checkout_date': pd.Timestamp('2024-01-01') + pd.Timedelta(days=np.random.randint(0, 90))
    }
    checkouts_data.append(checkout)

checkouts_df = pd.DataFrame(checkouts_data)
print(f"Generated {len(students_df)} students across {len(schools)} schools")
print(f"Generated {len(checkouts_df)} checkout events")
print(f"Average checkouts per student: {len(checkouts_df) / len(students_df):.1f}")

# Show checkout distribution
checkout_counts = checkouts_df['student_id'].value_counts()
print(f"Checkout distribution - Min: {checkout_counts.min()}, Max: {checkout_counts.max()}, Median: {checkout_counts.median():.1f}")

# Fulton County Reading Lift - Hybrid Recommendation System Demo

This notebook demonstrates the hybrid recommendation approach outlined in the proposal:
1. **Collaborative Filtering** - Find books borrowed by similar students
2. **Content-Based Similarity** - Recommend books with related themes/genres
3. **Policy Guardrails** - Enforce grade-band and suitability constraints
4. **LLM Explanations** - Generate understandable recommendation reasons

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

print("Setting up hybrid recommendation system...")

## 1. Generate Synthetic Library Data

Creating realistic synthetic data representing Fulton County middle school libraries.

In [ ]:
# Generate synthetic book catalog
np.random.seed(42)

book_titles = [
    "The Giver", "Wonder", "Hatchet", "Bridge to Terabithia", "The Outsiders",
    "Holes", "Where the Red Fern Grows", "Number the Stars", "The Lightning Thief",
    "Harry Potter and the Sorcerer's Stone", "Diary of a Wimpy Kid", "The Hunger Games",
    "Divergent", "The Maze Runner", "Miss Peregrine's Home for Peculiar Children",
    "The Book Thief", "Ghost", "New Kid", "The Wild Robot", "Refugee",
    "Restart", "Fish in a Tree", "The War That Saved My Life", "Brown Girl Dreaming",
    "The Crossover", "Inside Out and Back Again", "Esperanza Rising", "Freak the Mighty",
    "Walk Two Moons", "Maniac Magee", "Island of the Blue Dolphins", "A Wrinkle in Time",
    "The Lion, the Witch and the Wardrobe", "Charlotte's Web", "Stuart Little",
    "The Secret Garden", "Matilda", "James and the Giant Peach", "The BFG",
    "Tales of a Fourth Grade Nothing", "Fudge-a-Mania", "Are You There God? It's Me, Margaret",
    "Blubber", "Deenie", "Forever", "Judy Moody", "Captain Underpants", "Dog Man",
    "Wings of Fire: The Dragonet Prophecy", "Dork Diaries"
]

genres = ['Fantasy', 'Realistic Fiction', 'Adventure', 'Mystery', 'Historical Fiction', 'Science Fiction', 'Humor']
grade_bands = ['6-7', '7-8', '6-8', '8-9']
reading_levels = ['Below Grade', 'On Grade', 'Above Grade']

# Create book catalog
books_data = []
for i, title in enumerate(book_titles):
    book = {
        'book_id': f'book_{i+1:03d}',
        'title': title,
        'author': f'Author {i+1}',
        'genre': np.random.choice(genres),
        'grade_band': np.random.choice(grade_bands),
        'reading_level': np.random.choice(reading_levels),
        'description': f'A compelling {np.random.choice(genres).lower()} story about {title.lower()}.'
    }
    books_data.append(book)

catalog_df = pd.DataFrame(books_data)
print(f"Generated catalog with {len(catalog_df)} books")
print(f"Genres: {catalog_df['genre'].unique()}")
print(f"Grade bands: {catalog_df['grade_band'].unique()}")
catalog_df.head()